# GNN for edge classification to link the tracksters

Here, we will create an Edge classification graph that will link the tracksters belonging to the same particle. We will have the following properties:
1. Nodes: Tracksters
2. Node features: Trackster features
3. True label: From Associations tree
4. Edge features: .....

## Extracting data for all particles

In [1]:
import uproot
import awkward as ak
import numpy as np

In [2]:
particle_files={
    "n":"flat_tree_for_Neutron.root",
    "p":"flat_tree_for_Proton.root",
    "Y":"flat_tree_for_Photon.root",
    "pi0":"flat_tree_for_pion0.root",
    "pi+":"flat_tree_for_pion.root"
}
particle_ids={
    "n":0,
    "p":1,
    "Y":2,
    "pi0":3,
    "pi+":4
}

In [ ]:
%%time
all_events=[]
all_labels=[]
for pname,file_path in particle_files.items():
    file=uproot.open(file_path)
    #opening the trees
    reco=file["ticlDumper/CLUE3DHighTree"].arrays(library="ak")[:1000]
    sim=file["ticlDumper/SimTrackstersTree"].arrays(library="ak")[:1000]
    assoc=file["ticlDumper/associations"].arrays(library="ak")[:1000]
    simTICL=file["ticlDumper/simTICLCandidate"].arrays(library="ak")[:1000]
    #Particle-IDs
    pid=particle_ids[pname]
    n_events=len(reco)
    all_events.extend([
        (
            {
                "reco":reco[i],
                "sim":sim[i],
                "assoc":assoc[i],
                "simTICL":simTICL[i]
            },
            pid
        )
        for i in range(n_events)
    ])

In [ ]:
len(all_events)

In [ ]:
all_events[1000][0]["reco"]

In [ ]:
all_events[0:5]

In [ ]:
all_events[0][0]['reco']

### Shuffling

In [ ]:
from sklearn.utils import shuffle
all_events=shuffle(all_events,random_state=28)

In [ ]:
sample=all_events[1]
event,pid=sample

In [ ]:
event

In [ ]:
pid

In [ ]:
reco_dat=event["reco"]

In [ ]:
reco_dat["barycenter_x"]

## Extracting the graph features

### Getting the node features

The tracksters are the nodes and the node features are the characteristics of the tracksters

In [ ]:
def get_node_features(event):
    
    feats=np.stack([
        event["time"],
        event["raw_energy"],
        event["raw_em_energy"],
        event["raw_pt"],
        event["barycenter_x"],
        event["barycenter_y"],
        event["barycenter_z"],
        event["barycenter_eta"],
        event["barycenter_phi"],
        event["EV1"],
        event["EV2"],
        event["EV3"],
        event["sigmaPCA1"],
        event["sigmaPCA2"],
        event["sigmaPCA3"]
    ],axis=1)
    
    return np.array(feats)

In [ ]:
#Testing
test_sample=all_events[99]
event,pid=test_sample
reco=event['reco']
node_features=get_node_features(reco)
node_features.shape

### Building the graph edges

In [ ]:
from sklearn.neighbors import NearestNeighbors

def build_edges(event,k=8):
    x=np.array(event["barycenter_x"])
    y=np.array(event["barycenter_y"])
    z=np.array(event["barycenter_z"])
    coords=np.stack([x,y,z],axis=1)
    # Fit KNN-determine the KNN
    n_nodes=len(coords)
    k_eff=min(k,n_nodes-1)
    if n_nodes<2:
        return np.empty((2,0),dtype=int)
    knn=NearestNeighbors(n_neighbors=k_eff+1,algorithm='auto').fit(coords)
    dist,idx=knn.kneighbors(coords)
    #Creating edges
    edges=[]
    for i,nbrs in enumerate(idx):
        for j in nbrs[1:]:
            edges.append([i,j])
    edges=np.array(edges).T
    return edges

In [ ]:
def build_edges(event):
    x=np.array(event["barycenter_x"])
    n_nodes=len(x)
    if n_nodes<2:
        return np.empty((2,0),dtype=int)
    #Creating the pairs of source and destination
    src=[]
    dst=[]
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i!=j:
                src.append(i)
                dst.append(j)
    edge_index=np.array([src,dst],dtype=int)
    return edge_index

In [ ]:
#Testing
test_sample=all_events[99]
event,pid=test_sample
reco=event['reco']
edge_index=build_edges(reco)
edge_index.shape

In [ ]:
src,dst=edge_index

In [ ]:
print(src)

### Build Edge Features

In [ ]:
def build_edge_features(event,edge_index):
    x=np.array(event["barycenter_x"])
    y=np.array(event["barycenter_y"])
    z=np.array(event["barycenter_z"])
    eta=np.array(event["barycenter_eta"])
    phi=np.array(event["barycenter_phi"])
    t=np.array(event["time"])
    E=np.array(event["raw_energy"])
    
    features=[]
    
    for i,j in edge_index.T:
        dx=x[i]-x[j]
        dy=y[i]-y[j]
        dz=z[i]-z[j]
        dist=np.sqrt(dx*dx+dy*dy+dz*dz)
        deta=eta[i]-eta[j]
        dphi=phi[i]-phi[j]
        dt=t[i]-t[j]
        dE=E[i]-E[j]
        features.append([dist,dz,deta,dphi,dt,dE])
    return np.array(features)

In [ ]:
#Testing
test_sample=all_events[99]
event,pid=test_sample
reco=event['reco']
node_features=get_node_features(reco)
edge_index=build_edges(reco)
edge_features=build_edge_features(reco,edge_index)

In [ ]:
edge_index.shape
edge_features.shape

### Getting the truth mappings

#### Choosing the threshold

In [ ]:
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt
all_ratios=[]
for event,pid in all_events:
    assoc=event["assoc"]
    simTICL=event["simTICL"]
    reco_to_sim=ak.flatten(assoc["CLUE3DHigh_recoToSim_CP"])
    sharedE=ak.flatten(assoc["CLUE3DHigh_recoToSim_CP_sharedE"],axis=-1)
    sim_energy=simTICL["simTICLCandidate_raw_energy"]
    ratio=sharedE/sim_energy[reco_to_sim]
    ratio_flat=ak.flatten(ratio,axis=None)
    all_ratios.append(ratio_flat)
all_ratios=ak.concatenate(all_ratios)
ratio_np=ak.to_numpy(all_ratios)

In [ ]:
threshold=0.4
mask=ratio_np>threshold
ratio_np=ratio_np[mask]

In [ ]:
plt.figure(figsize=(5,4))
plt.hist(ratio_np,bins=100,histtype='step')
plt.xlabel("sharedE/simE")
plt.ylabel("Counts")
plt.yscale("log")
plt.show()

### Getting the mappings

In [ ]:
def get_truth_mappings(event,threshold=0.3):
    assoc=event["assoc"]
    simTICL=event["simTICL"]
    reco_to_sim=assoc["CLUE3DHigh_recoToSim_CP"]
    sharedE=assoc["CLUE3DHigh_recoToSim_CP_sharedE"]
    sim_energy=simTICL["simTICLCandidate_raw_energy"]
    truth={}
    for i,matches in enumerate(reco_to_sim):
        if len(matches)==0:
            truth[i]=-1
            continue
        matches=np.array(matches)
        shared=np.array(sharedE[i])
        #Remove invalid-indices
        valid=matches>=0
        if np.sum(valid)==0:
            truth[i]=-1
            continue
        matches=matches[valid]
        shared=shared[valid]
        denom=sim_energy[matches]
        denom=np.where(denom==0,1e-8,denom)
        score=shared/denom
        #score=sharedE[i]/sim_energy[matches]
        best_idx=int(np.argmax(score))
        best_score=score[best_idx]
        if best_score>threshold:
            truth[i]=int(matches[best_idx])
        else:
            truth[i]=-1
    return truth

In [ ]:
#Testing
test_sample=all_events[10]
event,pid=test_sample
reco=event['reco']
node_features=get_node_features(reco)
edge_index=build_edges(reco)
edge_features=build_edge_features(reco,edge_index)
truth=get_truth_mappings(event)

In [ ]:
truth

### Building edge labels

In [ ]:
def build_edge_labels(edge_index,truth):
    labels=[]
    for i,j in edge_index.T:
        ti=truth.get(int(i),-1)
        tj=truth.get(int(j),-1)
        if ti!=-1 and tj!=-1 and ti==tj:
            labels.append(1)
        else:
            labels.append(0)
    return np.array(labels)

In [ ]:
#Testing
test_sample=all_events[99]
event,pid=test_sample
reco=event['reco']
node_features=get_node_features(reco)
edge_index=build_edges(reco)
edge_features=build_edge_features(reco,edge_index)
truth=get_truth_mappings(event)
edge_labels=build_edge_labels(edge_index,truth)
edge_labels.shape

## Building the graph object

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data,Dataset
from torch_geometric.loader import DataLoader

In [ ]:
def build_graph(sample):
    event,pid=sample
    reco=event["reco"]
    node_features=get_node_features(reco)
    edge_index=build_edges(reco)
    edge_attr=build_edge_features(reco,edge_index)
    truth=get_truth_mappings(event)
    labels=build_edge_labels(edge_index,truth)
    data=Data(x=torch.tensor(node_features,dtype=torch.float),
              edge_index=torch.tensor(edge_index,dtype=torch.long),
              edge_attr=torch.tensor(edge_attr,dtype=torch.float),
               y=torch.tensor(pid,dtype=torch.long)
             )
    data.edge_label=torch.tensor(labels,dtype=torch.float)
    return data

In [ ]:
test_sample=all_events[99]
build_graph(test_sample)

## Building the dataset

In [ ]:
%%time
graph=[]
#all_events=all_events[:1000]
for sample in all_events:
    p_graph=build_graph(sample)
    graph.append(p_graph)

In [ ]:
len(graph)

In [ ]:
graph[99]

## Splitting into training,Validation and Testing data

In [ ]:
from sklearn.model_selection import train_test_split
#First split: Train vs Temp(Val+Test)
train_data, temp_data= train_test_split(graph, test_size=0.3, random_state=42)

#Second split: Test vs Val
val_data, test_data= train_test_split(temp_data, test_size=0.7, random_state=42)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")
print(f"Test samples: {len(test_data)}")

## Using the dataloader

In [ ]:
from torch_geometric.loader import DataLoader
batch_size=32

train_loader=DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader=DataLoader(val_data, batch_size=batch_size, shuffle=True)
test_loader=DataLoader(test_data, batch_size=batch_size, shuffle=True)

## Implementing the GNN

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class Edge_Classifier(nn.Module):
    def __init__(self,input_dim,hidden_dim):
        super().__init__()
        #First GCN Layer
        self.conv1=GCNConv(input_dim,hidden_dim)
        #Stacking 6 Repeated blocks
        self.convs=nn.ModuleList()
        for i in range(6):
            self.convs.append(GCNConv(hidden_dim,hidden_dim))
        #Final layer
        self.final_conv=GCNConv(hidden_dim,hidden_dim)
        #Regularization
        self.dropout=nn.Dropout(p=0.1)
        #The Edge Classification MLP
        self.edge_mlp=nn.Sequential(
            nn.Linear(2*hidden_dim,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,1)
        )
    def forward(self,data):
        x,edge_index,batch=data.x,data.edge_index,data.batch
        #-----------Passing the GCN forward passes--------------#
        ##### First GCN ######
        x=self.conv1(x,edge_index)
        x=F.relu(x)
        x=self.dropout(x)
        ##### Six repeated convolutions #####
        for conv in self.convs:
            x=conv(x,edge_index)
            x=F.relu(x)
            x=self.dropout(x)
        ##### Final convolution #####
        x=self.final_conv(x,edge_index)
        node_embedding=x
        #------------ Edge Classification MLP -----------------#
        row,col=edge_index
        edge_features=torch.cat([x[row],x[col]],dim=1)
        edge_logits=self.edge_mlp(edge_features).view(-1)
        return edge_logits

## Implementing the Edge Convolution network

In [ ]:
import torch
from torch.nn import Sequential as Seq, Linear, ReLU
from torch_geometric.nn import MessagePassing

class EdgeAttrConv(MessagePassing):
    def __init__(self,in_channels,edge_channels,out_channels):
        super().__init__(aggr='max')
        #The new MLP must take into input- node attributes, as well as edge attributes
        mlp_in_channels=(2*in_channels)+edge_channels
        self.mlp=Seq(Linear(mlp_in_channels,out_channels),
                     ReLU(),
                     Linear(out_channels,out_channels))
    def forward(self,x,edge_index,edge_attr):
        return self.propagate(edge_index,x=x,edge_attr=edge_attr)
    def message(self,x_i,x_j,edge_attr):
        #print("x_i shape:",x_i.shape)
        #print("x_j shape:",x_j.shape)
        #print("edge_attr shape:", edge_attr.shape)
        tmp=torch.cat([x_i,x_i-x_j,edge_attr],dim=-1)
        #print("tmp shape:", tmp.shape)
        #print("Expected MLP input:", self.mlp[0].in_features)
        return self.mlp(tmp)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class Edge_Classifier(nn.Module):
    def __init__(self,node_in,edge_in,hidden_dim):
        super().__init__()
        self.conv1=EdgeAttrConv(in_channels=node_in,edge_channels=edge_in,out_channels=hidden_dim)
        self.conv2=EdgeAttrConv(in_channels=hidden_dim,edge_channels=edge_in,out_channels=hidden_dim)
        self.edge_mlp=nn.Sequential(
            nn.Linear(2*hidden_dim+edge_in,hidden_dim),
            nn.ReLU(),
            #nn.Linear(2*hidden_dim,hidden_dim),
            #nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2,1)
        )
    def forward(self,data):
        x,edge_index,edge_attr,batch=data.x,data.edge_index,data.edge_attr,data.batch
        x1=self.conv1(x,edge_index,edge_attr)
        x2=self.conv2(x1,edge_index,edge_attr)
        node_embeddings=x2
        src,dst=edge_index
        edge_feat=torch.cat([x2[src],x2[dst],edge_attr],dim=-1)
        out_logits=self.edge_mlp(edge_feat)
        return node_embeddings,out_logits

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class Edge_Classifier(nn.Module):
    def __init__(self,node_in,edge_in,hidden_dim):
        super().__init__()
        self.conv1=EdgeAttrConv(in_channels=node_in,edge_channels=edge_in,out_channels=hidden_dim)
        self.conv2=EdgeAttrConv(in_channels=hidden_dim,edge_channels=edge_in,out_channels=hidden_dim)
        self.edge_mlp=nn.Sequential(
            nn.Linear(2*hidden_dim+edge_in,hidden_dim),
            nn.ReLU(),
            #nn.Linear(2*hidden_dim,hidden_dim),
            #nn.ReLU(),
            nn.Linear(hidden_dim,hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2,1)
        )
    def forward(self,data):
        x,edge_index,edge_attr,batch=data.x,data.edge_index,data.edge_attr,data.batch
        #Deubg- Input
        #if torch.isnan(x).any():
        #    print("Nan before conv1")
        #if torch.isnan(edge_attr).any():
        #    print("NaN BEFORE conv1 (edge_attr)")
        x1=self.conv1(x,edge_index,edge_attr)
        #if torch.isnan(x1).any():
        #    print(" NaN AFTER conv1")
        #    print("x1 max:", x1.abs().max().item())
        #    return x1, torch.zeros(edge_index.size(1), 1, device=x.device)
        x2=self.conv2(x1,edge_index,edge_attr)
        #if torch.isnan(x2).any():
        #    print(" NaN AFTER conv2")
        #    print("x2 max:", x2.abs().max().item())
        #    return x2, torch.zeros(edge_index.size(1), 1, device=x.device)
        node_embeddings=x2
        src,dst=edge_index
        edge_feat=torch.cat([x2[src],x2[dst],edge_attr],dim=-1)
        out_logits=self.edge_mlp(edge_feat)
        return node_embeddings,out_logits

In [ ]:
# Device setup
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ",device)

In [ ]:
node_dim=graph[0].x.shape[1]
edge_dim=graph[0].edge_attr.shape[1]
model=Edge_Classifier(node_in=node_dim,edge_in=edge_dim,hidden_dim=64).to(device)

In [ ]:
print(model)

In [ ]:
# Debug the model
test_batch=next(iter(train_loader)).to(device)
model(test_batch)

## Defining the Loss function

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#--------1. Focal Loss--------------#
class FocalLoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,logits,targets):
        y=targets.float().view(-1,1)
        p=torch.sigmoid(logits)
        eps=1e-8
        p=torch.clamp(p,eps,1-eps)
        loss=(-0.98*y*(1-p)**2*torch.log(p))-(0.02*(1-y)*(p**2)*torch.log(1-p))
        return loss.mean()

#----------2. Contrastive Loss-------------#
class ContrastiveLoss(nn.Module):
    def __init__(self,margin=2.0):
        super().__init__()
        self.margin=margin
    def forward(self,x,edge_index,targets):
        src,dst=edge_index
        xi=x[src]
        xj=x[dst]
        d=torch.norm(xi-xj,dim=1)
        y=targets.float()
        m=self.margin
        loss=y*d.pow(2)+((1-y)*torch.clamp(m-d,min=0).pow(2))
        return loss.mean()

#---------- Combined loss ---------------#
class CombinedObjective(nn.Module):
    def __init__(self):
        super().__init__()
        self.FL=FocalLoss()
        self.CL=ContrastiveLoss(margin=2.0)
    def forward(self,logits,targets,x,edge_index):
        FL=self.FL(logits,targets)
        CL=self.CL(x,edge_index,targets)
        total_loss=(100*FL)+(0.001*CL)
        return total_loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

#--------1. Focal Loss--------------#
class FocalLoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,logits,targets):
        y=targets.float().view(-1,1)
        p=torch.sigmoid(logits)
        eps=1e-8
        p=torch.clamp(p,eps,1-eps)
        loss=(-0.98*y*(1-p)**2*torch.log(p))-(0.02*(1-y)*(p**2)*torch.log(1-p))
        return loss.mean()

#----------2. Contrastive Loss-------------#
class ContrastiveLoss(nn.Module):
    def __init__(self,margin=2.0):
        super().__init__()
        self.margin=margin
    def forward(self,x,edge_index,targets):
        src,dst=edge_index
        xi=x[src]
        xj=x[dst]
        d=torch.norm(xi-xj,dim=1)+1e-8
        y=targets.float()
        m=self.margin
        loss=y*d.pow(2)+((1-y)*torch.clamp(m-d,min=0).pow(2))
        return loss.mean()

#---------- Combined loss ---------------#
class CombinedObjective(nn.Module):
    def __init__(self):
        super().__init__()
        self.FL=FocalLoss()
        self.CL=ContrastiveLoss(margin=2.0)
    def forward(self,logits,targets,x,edge_index):
        FL=self.FL(logits,targets)
        CL=self.CL(x,edge_index,targets)
        total_loss=(100*FL)+(0.001*CL)
        return total_loss

## Training, Testing and Validation

In [ ]:
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
criterion=CombinedObjective()

### Training

In [ ]:
# Traiing loop
def train():
    model.train()
    total_loss=0.0
    for batch in train_loader:
        batch=batch.to(device)
        optimizer.zero_grad()
        #Forward pass
        x,logits=model(batch)
        #Compute Loss
        loss=criterion(logits,batch.edge_label.view(-1),x,batch.edge_index)
        #Backpropagation
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(train_loader)

In [ ]:
# Traiing loop with debugging
def train():
    model.train()
    total_loss=0.0
    for batch in train_loader:
        batch=batch.to(device)
        #Debug-1- before model
        if torch.isnan(batch.x).any():
            print("NaN in node_features")
            break
        if torch.isnan(batch.edge_attr).any():
            print("NaN in edge_attr")
            break
        optimizer.zero_grad()
        #Forward pass
        x,logits=model(batch)
        #Debug-2- after model
        if torch.isnan(x).any():
            print("NaN in node embeddings")
            break
        if torch.isnan(logits).any():
            print("NaN in logits")
            break
        #Compute Loss
        loss=criterion(logits,batch.edge_label.view(-1),x,batch.edge_index)
        #Debug-3- Debugging Loss
        if torch.isnan(loss):
            print("NaN in loss")
            print("logits max:",logits.abs().max().item())
            print("embeddings max:",x.abs().max().item())
            break
        #Backpropagation
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    return total_loss/len(train_loader)

In [ ]:
# Validation Loop
def validate():
    model.eval()
    total_loss=0.0
    with torch.no_grad():
        for batch in val_loader:
            batch=batch.to(device)
            x,logits=model(batch)
            if torch.isnan(x).any() or torch.isnan(logits).any():
                print("NaN in validation forward")
                break
            loss=criterion(logits,batch.edge_label.view(-1),x,batch.edge_index)
            if torch.isnan(loss):
                print("NaN in validation loss")
                break
            total_loss+=loss.item()
    return total_loss/len(val_loader)

In [ ]:
%%time
epochs=10
train_losses=[]
val_losses=[]
for epoch in range(1,epochs+1):
    train_loss=train()
    val_loss=validate()
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch}, Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f}")

In [ ]:
#Visualizing the model performance
import matplotlib.pyplot as plt
plt.figure(figsize=(8,6))
plt.plot(range(1,len(train_losses)+1),train_losses,label="Training loss",color="Blue")
plt.plot(range(1,len(val_losses)+1),val_losses,label="Validation loss",color="Red")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation loss vs Epochs")
plt.legend()
plt.grid()
plt.show()

## Estimating model performance

In [ ]:
all_scores=[]
all_labels=[]
all_pids=[]
model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch=batch.to(device)
        node_emb,logits=model(batch)
        probs=torch.sigmoid(logits).view(-1)
        labels=batch.edge_label.view(-1)
        pids=batch.y[batch.batch]
        edge_pids=pids[batch.edge_index[0]]
        
        all_scores.append(probs.cpu())
        all_labels.append(labels.cpu())
        all_pids.append(edge_pids.cpu())

In [ ]:
scores=torch.cat(all_scores).numpy()
labels=torch.cat(all_labels).numpy()
pids=torch.cat(all_pids).numpy()

### Edge Score Distribution

In [ ]:
#Plotting for all particles
import matplotlib.pyplot as plt

true_scores=scores[labels==1]
false_scores=scores[labels==0]

plt.figure()

plt.hist(true_scores,bins=50,alpha=0.6,label="True edges",density=True)
plt.hist(false_scores,bins=50,alpha=0.6,label="False edges",density=True)
plt.legend(loc='best')
#plt.yscale("log")

In [ ]:
# For each particle
particle_ids
for pname,pid in particle_ids.items():
    mask=(pids==pid)
    pid_scores=scores[mask]
    pid_labels=labels[mask]
    true_scores=pid_scores[pid_labels==1]
    false_scores=pid_scores[pid_labels==0]
    plt.figure()
    plt.hist(true_scores,bins=50,histtype='step',alpha=0.6,label="True edges",density=True)
    plt.hist(false_scores,bins=50,histtype='step',alpha=0.6,label="False edges",density=True)
    plt.legend(loc='best')
    plt.title(f"Edge-Classification-{pname}")
    plt.show()

### ROC-AUC curve

In [ ]:
from sklearn.metrics import roc_curve,auc
fpr,tpr,thresholds=roc_curve(labels,scores)
roc_auc=auc(fpr,tpr)

In [ ]:
plt.plot(fpr,tpr)

In [ ]:
# For each particle
particle_ids
for pname,pid in particle_ids.items():
    mask=(pids==pid)
    pid_scores=scores[mask]
    pid_labels=labels[mask]
    # Skip if not enough data
    #if len(np.unique(pid_labels)) < 2:
    #    print(f"Skipping {pname} (only one class present)")
    #    continue
    fpr,tpr,_=roc_curve(pid_labels,pid_scores)
    roc_auc=auc(fpr,tpr)
    plt.figure()
    plt.plot(fpr,tpr,label=f"AUC {roc_auc:.3f}")
    plt.plot([0,1],[0,1],linestyle='--')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title(f"ROC Curve -{pname}")
    plt.legend()

    plt.show()

In [ ]:
print(torch.__version__)